In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import glob

# --- CONFIG ---
# Update this path to where you moved the files
FOLDER_PATH = Path("../data/raw/sdr_captures") 
SAMPLE_RATE = 100e6 
DTYPE = np.float32

# Find all bin files
files = sorted(list(FOLDER_PATH.glob("*.bin")))
print(f"Found {len(files)} files to inspect.")

def analyze_file(file_path, file_index):
    print(f"\n--- Analyzing: {file_path.name} ---")
    
    # 1. Load a small slice (first 10ms is enough to see the spectrum)
    # 10ms * 100M samples/sec = 1,000,000 samples
    slice_len = 1_000_000 
    
    try:
        # Map file and take slice
        raw = np.memmap(file_path, dtype=DTYPE, mode='r')
        n_total = len(raw) // 2
        
        # Safety check: is file big enough?
        if n_total < slice_len:
            print("  Warning: File is shorter than 10ms!")
            current_slice_len = n_total
        else:
            current_slice_len = slice_len
            
        # Extract I/Q
        # Grab 2x length because of interleaved I,Q
        chunk = raw[:current_slice_len*2]
        i_data = chunk[0::2]
        q_data = chunk[1::2]
        complex_data = i_data + 1j * q_data
        
        # 2. Calculate Stats
        power = np.abs(complex_data)**2
        mean_pwr = np.mean(power)
        max_pwr = np.max(power)
        
        # 3. Create Report Card Plot
        fig, axes = plt.subplots(2, 2, figsize=(14, 8))
        fig.suptitle(f"File: {file_path.name}\n(Mean Power: {mean_pwr:.4f} | Max: {max_pwr:.4f})", fontsize=14)
        
        # A) Time Domain (Zoomed in to first 500 samples)
        axes[0,0].plot(i_data[:500], label='I', alpha=0.7)
        axes[0,0].plot(q_data[:500], label='Q', alpha=0.7)
        axes[0,0].set_title("Time Domain (First 500 samples)")
        axes[0,0].legend()
        
        # B) Power Spectrum (PSD) - Shows Frequency Activity
        axes[0,1].psd(complex_data, NFFT=1024, Fs=SAMPLE_RATE, color='purple')
        axes[0,1].set_title("Power Spectral Density (The Fingerprint)")
        
        # C) Histogram - Shows Statistical Distribution
        axes[1,0].hist(i_data, bins=100, alpha=0.5, label='I', color='blue')
        axes[1,0].hist(q_data, bins=100, alpha=0.5, label='Q', color='orange')
        axes[1,0].set_title("Value Distribution")
        axes[1,0].legend()
        
        # D) Spectrogram (Waterfalls)
        axes[1,1].specgram(complex_data, NFFT=1024, Fs=SAMPLE_RATE, noverlap=512)
        axes[1,1].set_title("Spectrogram (Time vs Freq)")
        
        plt.tight_layout()
        plt.show()
        
        # Clean up memmap
        del raw
        
    except Exception as e:
        print(f"FAILED to read {file_path.name}: {e}")

# Run loop
for i, f in enumerate(files):
    analyze_file(f, i)